# Camada Silver - Transformação e Limpeza

Processa todas as 6 tabelas da camada Bronze aplicando: tipagem correta, formatação de moeda em R$ (padrão brasileiro), normalização de timestamps para UTC-3 (São Paulo) e persistência em tabelas Delta na camada Silver.

In [0]:
from pyspark.sql.functions import *

# Criar schema silver se não existir
spark.sql("CREATE DATABASE IF NOT EXISTS silver")

# Funções reutilizáveis

def format_currency_br(df, col_name):
    """Formata coluna decimal no padrão brasileiro: R$ 1.981,30 ou -R$ 50,00"""
    br_value = (
        regexp_replace(
            regexp_replace(
                regexp_replace(
                    format_number(abs(col(col_name)), 2),
                    ",", "§"
                ),
                "\\.", ","
            ),
            "§", "."
        )
    )
    return (
        df.withColumn(
            f"{col_name}_br",
            when(col(col_name) < 0, concat(lit("-R$ "), br_value))
            .otherwise(concat(lit("R$ "), br_value))
        )
        .drop(col_name)
        .withColumnRenamed(f"{col_name}_br", col_name)
    )

def normalize_timestamp_sp(df, col_name="_ingested_at"):
    """Converte timestamp UTC para horário de São Paulo (America/Sao_Paulo)"""
    return df.withColumn(
        col_name,
        from_utc_timestamp(to_timestamp(col(col_name)), "America/Sao_Paulo")
    )

def move_column_to_end(df, col_name):
    """Move uma coluna para o final do DataFrame"""
    cols = [c for c in df.columns if c != col_name]
    return df.select(*cols, col_name)

In [0]:
# Transformações específicas por tabela

def transform_payments(df):
    return (
        df.drop("order_id")
        .withColumn("payment_id", regexp_replace(col("payment_id"), "^PAY0*", ""))
        .withColumn("payment_type", lower(trim(col("payment_type"))))  # pix/PIX → pix
        .withColumn("payment_value", expr("try_cast(payment_value as decimal(10,2))"))
        .transform(lambda d: format_currency_br(d, "payment_value"))
    )

def transform_customers(df):
    return (
        df.withColumn("customer_id", regexp_replace(col("customer_id"), "^C0*", ""))
        .withColumn("email", when(col("email").isNull(), lit("não informado")).otherwise(col("email")))
        .withColumn("state", when(col("state").isNull(), lit("desconhecido")).otherwise(col("state")))
        .withColumn("signup_date", to_date(col("signup_date")))
    )

def transform_orders(df):
    return (
        df.withColumn("order_id", regexp_replace(col("order_id"), "^O0*", ""))
        .withColumn("customer_id", when(col("customer_id").isNull(), lit("desconhecido"))
        .otherwise(regexp_replace(col("customer_id"), "^C0*", "")))
        .withColumn("order_date", to_date(col("order_date")))
        .withColumn("status", lower(trim(col("status"))))  # DELIVERED → delivered
        .withColumn("total_amount", expr("try_cast(total_amount as decimal(10,2))"))
        .transform(lambda d: format_currency_br(d, "total_amount"))
    )

def transform_order_items(df):
    return (
        df.withColumn("order_id", regexp_replace(col("order_id"), "^O0*", ""))
        .withColumn("item_id", regexp_replace(col("item_id"), "^O0*", ""))
        .withColumn("product_id", regexp_replace(col("product_id"), "^P0*", ""))
        .withColumn("quantity", expr("try_cast(quantity as int)"))
        .withColumn("unit_price", expr("try_cast(unit_price as decimal(10,2))"))
        .transform(lambda d: format_currency_br(d, "unit_price"))
    )

def transform_products(df):
    return (
        df.withColumn("product_id", regexp_replace(col("product_id"), "^P0*", ""))
        .withColumn("product_name", when(col("product_name").isNull(), lit("Produto sem nome"))
        .otherwise(col("product_name")))
        .withColumn("price", expr("try_cast(price as decimal(10,2))"))
        .withColumn("stock", expr("try_cast(stock as int)"))
        .transform(lambda d: format_currency_br(d, "price"))
    )

def transform_reviews(df):
    return (
        df.withColumn("review_id", regexp_replace(col("review_id"), "^R0*", ""))
        .withColumn("order_id", regexp_replace(col("order_id"), "^O0*", ""))
        .withColumn("customer_id", regexp_replace(col("customer_id"), "^C0*", ""))
        .withColumn("rating", expr("try_cast(rating as int)"))
        .withColumn("comment", when(col("comment").isNull(), lit("sem comentário"))
        .otherwise(col("comment")))
        .withColumn("review_date", to_date(col("review_date")))
    )

# Processar e salvar todas as tabelas
table_transforms = {
    "payments": transform_payments,
    "customers": transform_customers,
    "orders": transform_orders,
    "order_items": transform_order_items,
    "products": transform_products,
    "reviews": transform_reviews,
}

for table_name, transform_fn in table_transforms.items():
    df = spark.table(f"bronze.{table_name}")
    df = transform_fn(df)
    df = normalize_timestamp_sp(df)
    df = move_column_to_end(df, "_ingested_at")
    
    df.write.format("delta").mode("overwrite").saveAsTable(f"silver.{table_name}")
    print(f"✓ silver.{table_name} | {df.count()} linhas | {len(df.columns)} colunas")

print("\nCamada Silver concluída!")

In [0]:
# Preview das primeiras linhas de cada tabela Silver
for table_name in ["payments", "customers", "orders", "order_items", "products", "reviews"]:
    print(f"\n{'='*50}")
    print(f"silver.{table_name}")
    print(f"{'='*50}")
    display(spark.table(f"silver.{table_name}").limit(10))